# Assignment 1 — Build a Custom Missing-Value Imputer

**Feature Engineering & MLOps · Unit 1, Session 4 follow-up (Missing Values)**

We implement `CustomImputer`, a scikit-learn-style transformer (`fit` / `transform`)
that fills missing values — mean or median for numeric columns, the mode for
categorical/text columns — with every statistic learned **only from the training
split**. It also optionally adds a `<column>_was_missing` indicator per column that had
missing data in training, which matters for the MNAR column `mock_test_3` (Session 4).

**Notebook layout**

* **Part 4.1** — the `CustomImputer` class
* **Part 4.2** — apply it to the PrepEdge dataset, with verification and a SimpleImputer
  cross-check
* **Part 4.3** — reflection questions
* **Bonus** — per-column strategy overrides

In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.utils.validation import check_is_fitted

RANDOM_STATE = 42
DATA_PATH = "../data/raw/student_performance_raw.csv"
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

## Part 4.1 — The `CustomImputer` class

Design decisions, made explicit:

* **`fit` computes a fill value for *every* column** (mean/median if numeric, mode if
  not) and stores them in `self.fill_values_`. Computing them for all columns — not just
  the ones missing in training — means the transformer can still handle a column that
  happens to be complete in train but has gaps in test.
* **`fit` also records `self.columns_with_missing_`** — the columns that actually had
  NaNs in the training data. Only these get a `_was_missing` indicator in `transform`,
  exactly as the assignment specifies.
* **`transform` never recomputes a statistic.** It only reads `self.fill_values_`, so
  train and test are filled with identical values (no leakage).
* **Column type is auto-detected** with `pandas.api.types.is_numeric_dtype`.
* **`transform` before `fit` raises** via `check_is_fitted`.

In [2]:
class CustomImputer(BaseEstimator, TransformerMixin):
    """Impute missing values with statistics learned only during ``fit``.

    Numeric columns are filled with their training mean or median; categorical /
    text columns are filled with their training mode. Optionally, a binary
    ``<column>_was_missing`` indicator is appended for every column that contained
    missing values in the training data (useful when missingness is informative,
    e.g. an MNAR column).

    Follows the scikit-learn transformer API: ``fit`` returns ``self`` and stores
    fitted state on attributes ending in ``_``; ``transform`` returns a new
    DataFrame and never recomputes statistics.

    Parameters
    ----------
    numeric_strategy : {"median", "mean"}, default="median"
        Statistic used to fill numeric columns.
    categorical_strategy : {"most_frequent"}, default="most_frequent"
        Statistic used to fill categorical / text columns. Only the mode is
        supported; the parameter exists for API symmetry and future extension.
    add_missing_indicator : bool, default=True
        If True, append one ``<column>_was_missing`` (0/1) column for each column
        that had missing values in the training data.

    Attributes
    ----------
    fill_values_ : dict
        Mapping ``{column_name: fill_value}`` learned during ``fit`` (one entry
        per column of the training frame).
    columns_with_missing_ : list of str
        Columns that contained at least one missing value in the training data.
    numeric_columns_ : list of str
        Columns detected as numeric during ``fit``.
    categorical_columns_ : list of str
        Columns detected as non-numeric (categorical / text) during ``fit``.
    feature_names_in_ : numpy.ndarray
        Column names seen during ``fit`` (scikit-learn convention).
    """

    def __init__(self, numeric_strategy="median",
                 categorical_strategy="most_frequent",
                 add_missing_indicator=True):
        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy
        self.add_missing_indicator = add_missing_indicator

    def fit(self, X, y=None):
        """Learn a fill value for every column of ``X``.

        Parameters
        ----------
        X : pandas.DataFrame
            Training data. Statistics are computed from these rows only.
        y : ignored
            Present for API compatibility.

        Returns
        -------
        self : CustomImputer
            The fitted transformer.
        """
        if not isinstance(X, pd.DataFrame):
            raise TypeError("CustomImputer expects a pandas DataFrame.")
        if self.numeric_strategy not in ("mean", "median"):
            raise ValueError(
                f"numeric_strategy must be 'mean' or 'median', "
                f"got {self.numeric_strategy!r}."
            )

        self.fill_values_ = {}
        self.columns_with_missing_ = []
        self.numeric_columns_ = []
        self.categorical_columns_ = []

        for col in X.columns:
            series = X[col]
            is_numeric = pd.api.types.is_numeric_dtype(series)

            if is_numeric:
                self.numeric_columns_.append(col)
                fill = series.mean() if self.numeric_strategy == "mean" else series.median()
            else:
                self.categorical_columns_.append(col)
                mode = series.mode(dropna=True)
                fill = mode.iloc[0] if not mode.empty else np.nan

            self.fill_values_[col] = fill
            if series.isna().any():
                self.columns_with_missing_.append(col)

        self.feature_names_in_ = np.asarray(X.columns)
        return self

    def transform(self, X):
        """Fill missing values using the statistics learned in ``fit``.

        Parameters
        ----------
        X : pandas.DataFrame
            Data to impute (train or test). Column names must match those seen
            during ``fit``.

        Returns
        -------
        X_out : pandas.DataFrame
            A copy of ``X`` with missing values filled. If
            ``add_missing_indicator=True``, one ``<column>_was_missing`` column is
            appended for each column that had missing values in the training data.

        Raises
        ------
        sklearn.exceptions.NotFittedError
            If called before ``fit``.
        """
        check_is_fitted(self, ["fill_values_", "columns_with_missing_"])
        if not isinstance(X, pd.DataFrame):
            raise TypeError("CustomImputer expects a pandas DataFrame.")

        X_out = X.copy()

        # Indicators first, computed from the ORIGINAL (pre-fill) values.
        if self.add_missing_indicator:
            for col in self.columns_with_missing_:
                if col in X_out.columns:
                    X_out[f"{col}_was_missing"] = X_out[col].isna().astype(int)

        # Fill every known column for which we have a usable fill value.
        for col, fill in self.fill_values_.items():
            if col in X_out.columns and pd.notna(fill):
                X_out[col] = X_out[col].fillna(fill)

        return X_out


### Guard rail: `transform` before `fit`

In [3]:
try:
    CustomImputer().transform(pd.DataFrame({"a": [1, np.nan, 3]}))
except Exception as e:
    print(f"{type(e).__name__}: {e}")

NotFittedError: This CustomImputer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.


## Part 4.2 — Apply to the PrepEdge dataset

### 4.2.1 — Load and make one 80 / 20 train / test split

We keep **all** columns (including the MNAR column `mock_test_3`) and split once, with a
fixed `random_state`, so every later step uses the same rows.

In [4]:
df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)

missing_summary = df.isna().sum()
print("\nColumns with missing values (full dataset):")
print(missing_summary[missing_summary > 0])

train_df, test_df = train_test_split(df, test_size=0.20, random_state=RANDOM_STATE)
print(f"\ntrain rows: {len(train_df)}   test rows: {len(test_df)}")

shape: (600, 17)

Columns with missing values (full dataset):
weekly_study_hours    36
income_bracket        30
prev_exam_score       25
mock_test_3           21
feedback_text         68
dtype: int64

train rows: 480   test rows: 120


The five columns with missing data match the assignment's Session-4 table:

| Column | Missingness | Type |
|---|---|---|
| `weekly_study_hours` | MCAR | numeric |
| `prev_exam_score` | MAR (linked to `attendance_pct`) | numeric |
| `mock_test_3` | MNAR (linked to its own low values) | numeric |
| `income_bracket` | — | categorical (ordinal) |
| `feedback_text` | — | text |

### 4.2.2 — Fit on the training split only, then transform both splits

In [5]:
imputer = CustomImputer(numeric_strategy="median", add_missing_indicator=True)
imputer.fit(train_df)                       # <-- training rows only

train_imputed = imputer.transform(train_df)
test_imputed = imputer.transform(test_df)

print("Columns detected as numeric     :", imputer.numeric_columns_)
print("Columns detected as categorical :", imputer.categorical_columns_)
print("\nColumns that had missing values in training:")
print(" ", imputer.columns_with_missing_)
print("\nLearned fill values for those columns:")
for c in imputer.columns_with_missing_:
    print(f"  {c:20s} -> {imputer.fill_values_[c]!r}")
print("\nIndicator columns added:",
      [c for c in train_imputed.columns if c.endswith("_was_missing")])

Columns detected as numeric     : ['student_id', 'city_tier', 'age', 'attendance_pct', 'weekly_study_hours', 'prev_exam_score', 'mock_test_1', 'mock_test_2', 'mock_test_3', 'doubt_sessions_attended', 'final_score']
Columns detected as categorical : ['city', 'course', 'batch_type', 'enrollment_date', 'income_bracket', 'feedback_text']

Columns that had missing values in training:
  ['weekly_study_hours', 'income_bracket', 'prev_exam_score', 'mock_test_3', 'feedback_text']

Learned fill values for those columns:
  weekly_study_hours   -> np.float64(5.0)
  income_bracket       -> '5-10L'
  prev_exam_score      -> np.float64(66.1)
  mock_test_3          -> np.float64(71.3)
  feedback_text        -> 'Need more practice sheets for weak topics'

Indicator columns added: ['weekly_study_hours_was_missing', 'income_bracket_was_missing', 'prev_exam_score_was_missing', 'mock_test_3_was_missing', 'feedback_text_was_missing']


### 4.2.3 — Verify no missing values remain

In [6]:
imputed_cols = [c for c in imputer.columns_with_missing_]

train_left = train_imputed[imputed_cols].isna().sum().sum()
test_left = test_imputed[imputed_cols].isna().sum().sum()
print(f"Missing values left in imputed columns -> train: {train_left}   test: {test_left}")

assert train_left == 0, "train still has missing values!"
assert test_left == 0, "test still has missing values!"
assert train_imputed.index.equals(train_df.index), "row order / index changed!"
assert len(train_imputed) == len(train_df), "row count changed!"
print("All assertions passed: no missing values, index and row count preserved.")

Missing values left in imputed columns -> train: 0   test: 0
All assertions passed: no missing values, index and row count preserved.


### 4.2.4 — Mean & standard deviation, before vs after imputation

In [7]:
num_missing_cols = [c for c in imputer.numeric_columns_
                    if c in imputer.columns_with_missing_]
print("Numeric columns that were imputed:", num_missing_cols)

rows = []
for col in num_missing_cols:
    before, after = train_df[col], train_imputed[col]
    rows.append({
        "column": col,
        "n_missing (train)": int(before.isna().sum()),
        "mean_before": before.mean(), "mean_after": after.mean(),
        "std_before": before.std(),  "std_after": after.std(),
    })
stats_tbl = pd.DataFrame(rows).set_index("column")
stats_tbl["mean_delta"] = stats_tbl["mean_after"] - stats_tbl["mean_before"]
stats_tbl["std_delta"] = stats_tbl["std_after"] - stats_tbl["std_before"]
stats_tbl

Numeric columns that were imputed: ['weekly_study_hours', 'prev_exam_score', 'mock_test_3']


,n_missing (train),mean_before,mean_after,std_before,std_after,mean_delta,std_delta
column,,,,,,,
weekly_study_hours,25,6.064,6.009,4.350,4.242,-0.055,-0.108
prev_exam_score,23,65.825,65.838,14.371,14.022,0.013,-0.349
mock_test_3,15,70.702,70.720,19.077,18.777,0.019,-0.301


**What changed.**

* **The mean barely moves.** We fill with the median, and for these roughly-symmetric
  columns the median sits close to the mean, so adding copies of it leaves the average
  almost unchanged.
* **The standard deviation drops** for every column. Every imputed row now holds the
  exact same value (the median), which adds zero spread while increasing the count — so
  the variance, and hence the std, is pulled down. The size of the drop scales with how
  many values were filled and how wide the original spread was.
* This variance compression is the price of single-value imputation, and it is one
  reason the `_was_missing` indicator is worth keeping: it hands the model back a signal
  that the raw imputed column has hidden.

### 4.2.5 — Cross-check the fill values against scikit-learn's `SimpleImputer`

`SimpleImputer(strategy="median")` on the same training columns must produce the same
numbers our class stored in `fill_values_`. This is the evidence that the core logic is
correct.

In [8]:
num_cols_all = imputer.numeric_columns_
sk = SimpleImputer(strategy="median").fit(train_df[num_cols_all])

check = pd.DataFrame({
    "CustomImputer": [imputer.fill_values_[c] for c in num_cols_all],
    "SimpleImputer": sk.statistics_,
}, index=num_cols_all)
check["abs_diff"] = (check["CustomImputer"] - check["SimpleImputer"]).abs()
check["match"] = check["abs_diff"] < 1e-9
print(check)
assert check["match"].all(), "fill values disagree with SimpleImputer!"
print("\nAll numeric fill values match SimpleImputer(strategy='median') exactly.")

                         CustomImputer  SimpleImputer  abs_diff  match
student_id                   1,282.000      1,282.000     0.000   True
city_tier                        1.000          1.000     0.000   True
age                             17.000         17.000     0.000   True
attendance_pct                  78.350         78.350     0.000   True
weekly_study_hours               5.000          5.000     0.000   True
prev_exam_score                 66.100         66.100     0.000   True
mock_test_1                     66.550         66.550     0.000   True
mock_test_2                     66.350         66.350     0.000   True
mock_test_3                     71.300         71.300     0.000   True
doubt_sessions_attended          4.000          4.000     0.000   True
final_score                     52.700         52.700     0.000   True

All numeric fill values match SimpleImputer(strategy='median') exactly.


In [9]:
# And the same check for a categorical column vs SimpleImputer(strategy='most_frequent')
sk_cat = SimpleImputer(strategy="most_frequent").fit(train_df[["income_bracket"]])
print("income_bracket mode -> CustomImputer:", repr(imputer.fill_values_["income_bracket"]),
      "| SimpleImputer:", repr(sk_cat.statistics_[0]))
assert imputer.fill_values_["income_bracket"] == sk_cat.statistics_[0]
print("Categorical fill value matches too.")

income_bracket mode -> CustomImputer: '5-10L' | SimpleImputer: '5-10L'
Categorical fill value matches too.


## Part 4.3 — Reflection questions

**1. Why must `fit()` be called only on the training split, never on the full dataset
or the test split?**

`fit()` learns the fill statistics (medians, modes), and those numbers become part of
the model's preprocessing. If they are computed from the full dataset or the test split,
information about the test rows leaks into the pipeline before evaluation — the imputed
training values are now nudged by data the model is not supposed to have seen. The test
score then overstates real-world performance, because at deployment time you genuinely
will not have future rows to compute statistics from. Fitting on the training split only
keeps the test split a faithful stand-in for unseen data, and mirrors how the transform
must run in production: learn once on history, apply blindly to new records.

**2. `mock_test_3` is MNAR. Does mean/median imputation genuinely solve the problem, and
what does `add_missing_indicator` add?**

No. `mock_test_3` is missing *because the underlying score is low*, so the observed
values are a biased, upward sample of the truth. Filling the gaps with the median of the
observed (already too-high) values pushes the column's distribution even further from
reality and can weaken or invert the real relationship with the target. Plain imputation
implicitly assumes the data is MCAR, which is exactly what MNAR is not. The
`mock_test_3_was_missing` indicator does not fix the bias, but it preserves the one
thing plain imputation destroys: *which rows were missing*. A downstream model can then
learn "missing here tends to mean a weaker student" directly from the 0/1 column,
recovering some of the signal that the missingness itself carried.

**3. A brand-new column, entirely missing in the training data but present in test — what
does the current implementation do, and what should a production version do?**

In `fit`, an all-NaN numeric column yields `mean()/median() = NaN`, and an all-NaN
categorical column yields an empty `mode()`; either way this class stores `fill_value =
NaN` for it. In `transform`, the `pd.notna(fill)` guard then skips that column, so its
missing values in the test split are **left as NaN** (no crash, but also no imputation)
— and any model downstream will choke on them. A production-grade version should detect
zero observed values at fit time and act deliberately: fall back to a configured default
(0 for counts, `"Unknown"` for categories), or flag the column for dropping, emit a
warning naming the column and its missing rate, and enforce a schema so an unexpected
all-missing column is caught before it ever reaches a model.

## Bonus — Per-column strategy overrides

`CustomImputerWithOverrides` adds a `column_overrides` dict so a specific column can use
a strategy different from the dataset-wide default. Everything else — auto-detection,
indicators, the fitted-state convention, the guard rail — is inherited unchanged.

In [10]:
class CustomImputerWithOverrides(CustomImputer):
    """``CustomImputer`` plus a per-column strategy override.

    Parameters
    ----------
    numeric_strategy, categorical_strategy, add_missing_indicator
        See :class:`CustomImputer`.
    column_overrides : dict, optional
        Mapping ``{column_name: strategy}`` where ``strategy`` is one of
        ``"mean"``, ``"median"`` or ``"most_frequent"``. A column listed here
        uses the given strategy instead of the dataset-wide default.
    """

    _VALID = {"mean", "median", "most_frequent"}

    def __init__(self, numeric_strategy="median",
                 categorical_strategy="most_frequent",
                 add_missing_indicator=True, column_overrides=None):
        super().__init__(numeric_strategy, categorical_strategy, add_missing_indicator)
        self.column_overrides = column_overrides

    def fit(self, X, y=None):
        """Learn a fill value per column, honouring ``column_overrides``."""
        if not isinstance(X, pd.DataFrame):
            raise TypeError("CustomImputer expects a pandas DataFrame.")
        overrides = self.column_overrides or {}
        bad = set(overrides.values()) - self._VALID
        if bad:
            raise ValueError(f"invalid override strategy/strategies: {sorted(bad)}")

        self.fill_values_ = {}
        self.columns_with_missing_ = []
        self.numeric_columns_ = []
        self.categorical_columns_ = []
        self.strategy_used_ = {}

        for col in X.columns:
            series = X[col]
            is_numeric = pd.api.types.is_numeric_dtype(series)
            (self.numeric_columns_ if is_numeric else self.categorical_columns_).append(col)

            strategy = overrides.get(col,
                                     self.numeric_strategy if is_numeric
                                     else self.categorical_strategy)
            self.strategy_used_[col] = strategy

            if strategy == "mean":
                fill = series.mean()
            elif strategy == "median":
                fill = series.median()
            else:  # most_frequent
                mode = series.mode(dropna=True)
                fill = mode.iloc[0] if not mode.empty else np.nan

            self.fill_values_[col] = fill
            if series.isna().any():
                self.columns_with_missing_.append(col)

        self.feature_names_in_ = np.asarray(X.columns)
        return self


In [11]:
ov = CustomImputerWithOverrides(
    numeric_strategy="median",                     # dataset-wide default
    column_overrides={
        "weekly_study_hours": "mean",              # override: numeric -> mean
        "mock_test_3": "mean",                     # override: numeric -> mean
    },
)
ov.fit(train_df)
_ = ov.transform(train_df); _ = ov.transform(test_df)

demo = pd.DataFrame({
    "strategy_used": {c: ov.strategy_used_[c] for c in
                      ["weekly_study_hours", "mock_test_3", "prev_exam_score"]},
    "fill_value": {c: ov.fill_values_[c] for c in
                   ["weekly_study_hours", "mock_test_3", "prev_exam_score"]},
})
print(demo)
print("\nweekly_study_hours: mean", round(train_df['weekly_study_hours'].mean(), 3),
      "vs median", round(train_df['weekly_study_hours'].median(), 3),
      "-> override correctly used the mean")
print("prev_exam_score kept the median default:",
      ov.fill_values_['prev_exam_score'] == train_df['prev_exam_score'].median())

                   strategy_used  fill_value
weekly_study_hours          mean       6.064
mock_test_3                 mean      70.702
prev_exam_score           median      66.100

weekly_study_hours: mean 6.064 vs median 5.0 -> override correctly used the mean
prev_exam_score kept the median default: True


`weekly_study_hours` and `mock_test_3` are filled with their **mean** (per the
override), while `prev_exam_score` still uses the **median** default — confirmed by the
printed fill values.